# PySpark Practice Notebook

Hands-on, runnable code practice for every concept in the main [`README.md`](../README.md), in the same order.
Run top to bottom — each section builds on DataFrames created earlier.

**Setup:** `pip install pyspark` (Java 11/17/21 must be installed). No cluster needed — this runs Spark locally.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("PySparkInterviewPrep")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")  # small for a local/demo run
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print(spark.version)

Picked up JAVA_TOOL_OPTIONS: -Djavax.net.ssl.trustStore=/root/.ccr/java-truststore.p12 -Djavax.net.ssl.trustStorePassword=changeit -Djavax.net.ssl.trustStoreType=PKCS12 -Dhttps.proxyHost=127.0.0.1 -Dhttps.proxyPort=37297 -Dhttp.nonProxyHosts=localhost|127.0.0.1|::1|127.*|0.*|::|169.254.*|api.anthropic.com|api-staging.anthropic.com|api-pr-preview.anthropic.com|mcp-proxy.anthropic.com|mcp-proxy-staging.anthropic.com|registry.npmjs.org|jsr.io|npm.jsr.io|pypi.org|files.pythonhosted.org|index.crates.io|proxy.golang.org|host.docker.internal|10.*|172.16.*|172.17.*|172.18.*|172.19.*|172.20.*|172.21.*|172.22.*|172.23.*|172.24.*|172.25.*|172.26.*|172.27.*|172.28.*|172.29.*|172.30.*|172.31.*|192.168.*|100.64.0.0/10|*.svc.cluster.local|*.svc.cluster.local -Djdk.http.auth.tunneling.disabledSchemes= -Djdk.http.auth.proxying.disabledSchemes=


Picked up JAVA_TOOL_OPTIONS: -Djavax.net.ssl.trustStore=/root/.ccr/java-truststore.p12 -Djavax.net.ssl.trustStorePassword=changeit -Djavax.net.ssl.trustStoreType=PKCS12 -Dhttps.proxyHost=127.0.0.1 -Dhttps.proxyPort=37297 -Dhttp.nonProxyHosts=localhost|127.0.0.1|::1|127.*|0.*|::|169.254.*|api.anthropic.com|api-staging.anthropic.com|api-pr-preview.anthropic.com|mcp-proxy.anthropic.com|mcp-proxy-staging.anthropic.com|registry.npmjs.org|jsr.io|npm.jsr.io|pypi.org|files.pythonhosted.org|index.crates.io|proxy.golang.org|host.docker.internal|10.*|172.16.*|172.17.*|172.18.*|172.19.*|172.20.*|172.21.*|172.22.*|172.23.*|172.24.*|172.25.*|172.26.*|172.27.*|172.28.*|172.29.*|172.30.*|172.31.*|192.168.*|100.64.0.0/10|*.svc.cluster.local|*.svc.cluster.local -Djdk.http.auth.tunneling.disabledSchemes= -Djdk.http.auth.proxying.disabledSchemes=


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/24 22:48:38 WARN Utils: Your hostname, vm, resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/08/24 22:48:38 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


/home/claude/pyspark-interview-prep/venv/lib/python3.11/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/08/24 22:48:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


4.2.0


## 2-3. SparkSession, reading data, and schema

In [2]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType

my_ddl_schema = "emp_id INT, name STRING, dept STRING, salary DOUBLE, join_date DATE, skills STRING"

df = (
    spark.read.format("csv")
    .option("header", True)
    .schema(my_ddl_schema)
    .load("../data/employees.csv")
)

df.printSchema()
df.show(5)

root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- dept: string (nullable = true)
 |-- salary: double (nullable = true)
 |-- join_date: date (nullable = true)
 |-- skills: string (nullable = true)



+------+------------+-----------+-------+----------+------------------+
|emp_id|        name|       dept| salary| join_date|            skills|
+------+------------+-----------+-------+----------+------------------+
|   101|    john doe|Engineering|95000.0|2021-03-15|  Python,Spark,SQL|
|   102|  jane smith|Engineering|88000.0|2020-07-01|        Java,Spark|
|   103|    alex kim|      Sales|62000.0|2022-01-10|         Excel,SQL|
|   104| priya patel|Engineering|95000.0|2019-11-20|Python,SQL,Airflow|
|   105|michael chen|      Sales|58000.0|2021-06-05|             Excel|
+------+------------+-----------+-------+----------+------------------+
only showing top 5 rows


In [3]:
# StructType version of the same schema (equivalent to the DDL string above)
my_struct_schema = StructType([
    StructField("emp_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("dept", StringType(), True),
    StructField("salary", DoubleType(), True),
    StructField("join_date", DateType(), True),
    StructField("skills", StringType(), True),
])

df_struct = (
    spark.read.format("csv")
    .option("header", True)
    .schema(my_struct_schema)
    .load("../data/employees.csv")
)
df_struct.printSchema()

root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- dept: string (nullable = true)
 |-- salary: double (nullable = true)
 |-- join_date: date (nullable = true)
 |-- skills: string (nullable = true)



## 4-5. SELECT and ALIAS

In [4]:
df.select("name", "salary").show(3)

df.select(col("name"), (col("salary") * 12).alias("annual_salary")).show(3)

+----------+-------+
|      name| salary|
+----------+-------+
|  john doe|95000.0|
|jane smith|88000.0|
|  alex kim|62000.0|
+----------+-------+
only showing top 3 rows


+----------+-------------+
|      name|annual_salary|
+----------+-------------+
|  john doe|    1140000.0|
|jane smith|    1056000.0|
|  alex kim|     744000.0|
+----------+-------------+
only showing top 3 rows


## 6. FILTER / WHERE (single `&` / `|` / `~`, not `and`/`or`/`not`)

In [5]:
df.filter(col("salary") > 80000).show()

df.filter((col("salary") > 80000) & (col("dept") == "Engineering")).show()

df.where((col("dept") == "Sales") | (col("dept") == "HR")).show()

df.filter(~(col("dept") == "HR")).show()

+------+-----------+-----------+--------+----------+--------------------+
|emp_id|       name|       dept|  salary| join_date|              skills|
+------+-----------+-----------+--------+----------+--------------------+
|   101|   john doe|Engineering| 95000.0|2021-03-15|    Python,Spark,SQL|
|   102| jane smith|Engineering| 88000.0|2020-07-01|          Java,Spark|
|   104|priya patel|Engineering| 95000.0|2019-11-20|  Python,SQL,Airflow|
|   107|     li wei|Engineering|120000.0|2018-09-01|Python,Spark,SQL,...|
+------+-----------+-----------+--------+----------+--------------------+



+------+-----------+-----------+--------+----------+--------------------+
|emp_id|       name|       dept|  salary| join_date|              skills|
+------+-----------+-----------+--------+----------+--------------------+
|   101|   john doe|Engineering| 95000.0|2021-03-15|    Python,Spark,SQL|
|   102| jane smith|Engineering| 88000.0|2020-07-01|          Java,Spark|
|   104|priya patel|Engineering| 95000.0|2019-11-20|  Python,SQL,Airflow|
|   107|     li wei|Engineering|120000.0|2018-09-01|Python,Spark,SQL,...|
+------+-----------+-----------+--------+----------+--------------------+

+------+------------+-----+-------+----------+--------------------+
|emp_id|        name| dept| salary| join_date|              skills|
+------+------------+-----+-------+----------+--------------------+
|   103|    alex kim|Sales|62000.0|2022-01-10|           Excel,SQL|
|   105|michael chen|Sales|58000.0|2021-06-05|               Excel|
|   106|  sara jones|   HR|54000.0|2023-02-14|       Communication|

+------+------------+-----------+--------+----------+--------------------+
|emp_id|        name|       dept|  salary| join_date|              skills|
+------+------------+-----------+--------+----------+--------------------+
|   101|    john doe|Engineering| 95000.0|2021-03-15|    Python,Spark,SQL|
|   102|  jane smith|Engineering| 88000.0|2020-07-01|          Java,Spark|
|   103|    alex kim|      Sales| 62000.0|2022-01-10|           Excel,SQL|
|   104| priya patel|Engineering| 95000.0|2019-11-20|  Python,SQL,Airflow|
|   105|michael chen|      Sales| 58000.0|2021-06-05|               Excel|
|   107|      li wei|Engineering|120000.0|2018-09-01|Python,Spark,SQL,...|
|   108| omar hassan|      Sales| 62000.0|2022-08-19|Excel,SQL,Negotia...|
|   110|daniel brown|Engineering| 79000.0|2023-05-23|          Python,SQL|
+------+------------+-----------+--------+----------+--------------------+



## 7-8. withColumnRenamed and withColumn

In [6]:
from pyspark.sql.functions import lit

df2 = df.withColumnRenamed("emp_id", "employee_id")
df2 = df2.withColumn("bonus", col("salary") * 0.10)
df2 = df2.withColumn("currency", lit("USD"))
df2.show(3)

+-----------+----------+-----------+-------+----------+----------------+------+--------+
|employee_id|      name|       dept| salary| join_date|          skills| bonus|currency|
+-----------+----------+-----------+-------+----------+----------------+------+--------+
|        101|  john doe|Engineering|95000.0|2021-03-15|Python,Spark,SQL|9500.0|     USD|
|        102|jane smith|Engineering|88000.0|2020-07-01|      Java,Spark|8800.0|     USD|
|        103|  alex kim|      Sales|62000.0|2022-01-10|       Excel,SQL|6200.0|     USD|
+-----------+----------+-----------+-------+----------+----------------+------+--------+
only showing top 3 rows


## 9. Type casting

In [7]:
df2 = df2.withColumn("salary_int", col("salary").cast(IntegerType()))
df2.printSchema()

root
 |-- employee_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- dept: string (nullable = true)
 |-- salary: double (nullable = true)
 |-- join_date: date (nullable = true)
 |-- skills: string (nullable = true)
 |-- bonus: double (nullable = true)
 |-- currency: string (nullable = false)
 |-- salary_int: integer (nullable = true)



## 10-13. SORT, LIMIT, DROP, DROP_DUPLICATES

In [8]:
df.sort(col("salary").desc()).show(3)

df.orderBy("dept", col("salary").desc()).show()

df.limit(3).show()

df.drop("skills").show(3)

# Two employees share salary=95000 -- see the difference vs a plain distinct()
df.select("salary").dropDuplicates().show()
df.dropDuplicates(["dept"]).select("dept", "name", "salary").show()

+------+-----------+-----------+--------+----------+--------------------+
|emp_id|       name|       dept|  salary| join_date|              skills|
+------+-----------+-----------+--------+----------+--------------------+
|   107|     li wei|Engineering|120000.0|2018-09-01|Python,Spark,SQL,...|
|   101|   john doe|Engineering| 95000.0|2021-03-15|    Python,Spark,SQL|
|   104|priya patel|Engineering| 95000.0|2019-11-20|  Python,SQL,Airflow|
+------+-----------+-----------+--------+----------+--------------------+
only showing top 3 rows
+------+------------+-----------+--------+----------+--------------------+
|emp_id|        name|       dept|  salary| join_date|              skills|
+------+------------+-----------+--------+----------+--------------------+
|   107|      li wei|Engineering|120000.0|2018-09-01|Python,Spark,SQL,...|
|   101|    john doe|Engineering| 95000.0|2021-03-15|    Python,Spark,SQL|
|   104| priya patel|Engineering| 95000.0|2019-11-20|  Python,SQL,Airflow|
|   102|

+------+----------+-----------+-------+----------+----------------+
|emp_id|      name|       dept| salary| join_date|          skills|
+------+----------+-----------+-------+----------+----------------+
|   101|  john doe|Engineering|95000.0|2021-03-15|Python,Spark,SQL|
|   102|jane smith|Engineering|88000.0|2020-07-01|      Java,Spark|
|   103|  alex kim|      Sales|62000.0|2022-01-10|       Excel,SQL|
+------+----------+-----------+-------+----------+----------------+



+------+----------+-----------+-------+----------+
|emp_id|      name|       dept| salary| join_date|
+------+----------+-----------+-------+----------+
|   101|  john doe|Engineering|95000.0|2021-03-15|
|   102|jane smith|Engineering|88000.0|2020-07-01|
|   103|  alex kim|      Sales|62000.0|2022-01-10|
+------+----------+-----------+-------+----------+
only showing top 3 rows


+--------+
|  salary|
+--------+
| 95000.0|
| 58000.0|
|120000.0|
| 88000.0|
| 62000.0|
| 54000.0|
| 79000.0|
| 57000.0|
+--------+



+-----------+----------+-------+
|       dept|      name| salary|
+-----------+----------+-------+
|Engineering|  john doe|95000.0|
|         HR|sara jones|54000.0|
|      Sales|  alex kim|62000.0|
+-----------+----------+-------+



## 14. UNION and UNION BY NAME

In [9]:
new_hires = spark.createDataFrame(
    [(201, "kevin wu", "Marketing", 67000.0, "2026-06-01", "SEO")],
    schema="emp_id INT, name STRING, dept STRING, salary DOUBLE, join_date STRING, skills STRING",
)
new_hires = new_hires.withColumn("join_date", col("join_date").cast(DateType()))

combined = df.union(new_hires)   # positional -- safe here because columns are already in the same order
combined.orderBy(col("emp_id").desc()).show(3)

# unionByName is safer in general -- demonstrate with reordered columns
reordered = new_hires.select("name", "emp_id", "dept", "salary", "join_date", "skills")
safe_union = df.unionByName(reordered)
safe_union.orderBy(col("emp_id").desc()).show(3)

+------+------------+-----------+-------+----------+--------------------+
|emp_id|        name|       dept| salary| join_date|              skills|
+------+------------+-----------+-------+----------+--------------------+
|   201|    kevin wu|  Marketing|67000.0|2026-06-01|                 SEO|
|   110|daniel brown|Engineering|79000.0|2023-05-23|          Python,SQL|
|   109|   grace lee|         HR|57000.0|2020-12-01|Communication,Rec...|
+------+------------+-----------+-------+----------+--------------------+
only showing top 3 rows
+------+------------+-----------+-------+----------+--------------------+
|emp_id|        name|       dept| salary| join_date|              skills|
+------+------------+-----------+-------+----------+--------------------+
|   201|    kevin wu|  Marketing|67000.0|2026-06-01|                 SEO|
|   110|daniel brown|Engineering|79000.0|2023-05-23|          Python,SQL|
|   109|   grace lee|         HR|57000.0|2020-12-01|Communication,Rec...|
+------+------

## 15. String functions

In [10]:
from pyspark.sql.functions import initcap, upper, lower, trim, concat_ws, substring

df.select(
    initcap("name").alias("proper_name"),
    upper("dept").alias("dept_upper"),
    substring("name", 1, 4).alias("first4_chars"),
).show(5)

+------------+-----------+------------+
| proper_name| dept_upper|first4_chars|
+------------+-----------+------------+
|    John Doe|ENGINEERING|        john|
|  Jane Smith|ENGINEERING|        jane|
|    Alex Kim|      SALES|        alex|
| Priya Patel|ENGINEERING|        priy|
|Michael Chen|      SALES|        mich|
+------------+-----------+------------+
only showing top 5 rows


## 16. DATEDIFF and DATE_FORMAT

Note the format pattern uses lowercase `yyyy` (calendar year), not `YYYY` (ISO week-year) -- a classic silent-bug source.

In [11]:
from pyspark.sql.functions import datediff, current_date, date_format

df.select(
    "name",
    "join_date",
    datediff(current_date(), col("join_date")).alias("days_employed"),
    date_format(col("join_date"), "yyyy-MM-dd").alias("formatted_date"),
    date_format(col("join_date"), "MMMM yyyy").alias("month_year"),
).show(5)

+------------+----------+-------------+--------------+-------------+
|        name| join_date|days_employed|formatted_date|   month_year|
+------------+----------+-------------+--------------+-------------+
|    john doe|2021-03-15|         1988|    2021-03-15|   March 2021|
|  jane smith|2020-07-01|         2245|    2020-07-01|    July 2020|
|    alex kim|2022-01-10|         1687|    2022-01-10| January 2022|
| priya patel|2019-11-20|         2469|    2019-11-20|November 2019|
|michael chen|2021-06-05|         1906|    2021-06-05|    June 2021|
+------------+----------+-------------+--------------+-------------+
only showing top 5 rows


## 17. Handling nulls

In [12]:
from pyspark.sql import Row

nulls_df = spark.createDataFrame([
    Row(emp_id=301, name="test one", dept=None, salary=50000.0),
    Row(emp_id=302, name=None, dept=None, salary=None),
    Row(emp_id=303, name="test three", dept="Sales", salary=None),
])

nulls_df.show()

print("dropna how='any':")
nulls_df.dropna(how="any").show()

print("dropna how='all' (only row 302 fully null gets dropped):")
nulls_df.dropna(how="all").show()

print("fillna with a dict of per-column defaults:")
nulls_df.fillna({"dept": "Unknown", "salary": 0.0, "name": "N/A"}).show()

+------+----------+-----+-------+
|emp_id|      name| dept| salary|
+------+----------+-----+-------+
|   301|  test one| NULL|50000.0|
|   302|      NULL| NULL|   NULL|
|   303|test three|Sales|   NULL|
+------+----------+-----+-------+

dropna how='any':


+------+----+----+------+
|emp_id|name|dept|salary|
+------+----+----+------+
+------+----+----+------+

dropna how='all' (only row 302 fully null gets dropped):


+------+----------+-----+-------+
|emp_id|      name| dept| salary|
+------+----------+-----+-------+
|   301|  test one| NULL|50000.0|
|   302|      NULL| NULL|   NULL|
|   303|test three|Sales|   NULL|
+------+----------+-----+-------+

fillna with a dict of per-column defaults:


+------+----------+-------+-------+
|emp_id|      name|   dept| salary|
+------+----------+-------+-------+
|   301|  test one|Unknown|50000.0|
|   302|       N/A|Unknown|    0.0|
|   303|test three|  Sales|    0.0|
+------+----------+-------+-------+



## 18-19. Split, indexing, explode, array_contains

In [13]:
from pyspark.sql.functions import split, explode, array_contains

df_arr = df.withColumn("skills_arr", split(col("skills"), ","))
df_arr.select("name", "skills_arr", col("skills_arr")[0].alias("first_skill")).show(5, truncate=False)

exploded = df_arr.select("name", explode("skills_arr").alias("skill"))
exploded.show(10)

df_arr.filter(array_contains(col("skills_arr"), "Spark")).select("name", "skills").show()

+------------+----------------------+-----------+
|name        |skills_arr            |first_skill|
+------------+----------------------+-----------+
|john doe    |[Python, Spark, SQL]  |Python     |
|jane smith  |[Java, Spark]         |Java       |
|alex kim    |[Excel, SQL]          |Excel      |
|priya patel |[Python, SQL, Airflow]|Python     |
|michael chen|[Excel]               |Excel      |
+------------+----------------------+-----------+
only showing top 5 rows


+-----------+-------+
|       name|  skill|
+-----------+-------+
|   john doe| Python|
|   john doe|  Spark|
|   john doe|    SQL|
| jane smith|   Java|
| jane smith|  Spark|
|   alex kim|  Excel|
|   alex kim|    SQL|
|priya patel| Python|
|priya patel|    SQL|
|priya patel|Airflow|
+-----------+-------+
only showing top 10 rows


+----------+--------------------+
|      name|              skills|
+----------+--------------------+
|  john doe|    Python,Spark,SQL|
|jane smith|          Java,Spark|
|    li wei|Python,Spark,SQL,...|
+----------+--------------------+



## 20. GROUP BY and COLLECT_LIST

In [14]:
from pyspark.sql.functions import sum as _sum, avg, count, collect_list, round as _round

df.groupBy("dept").agg(
    _sum("salary").alias("total_salary"),
    _round(avg("salary"), 2).alias("avg_salary"),
    count("*").alias("headcount"),
).orderBy(col("avg_salary").desc()).show()

df.groupBy("dept").agg(collect_list("name").alias("employees")).show(truncate=False)

+-----------+------------+----------+---------+
|       dept|total_salary|avg_salary|headcount|
+-----------+------------+----------+---------+
|Engineering|    477000.0|   95400.0|        5|
|      Sales|    182000.0|  60666.67|        3|
|         HR|    111000.0|   55500.0|        2|
+-----------+------------+----------+---------+

+-----------+---------------------------------------------------------+
|dept       |employees                                                |
+-----------+---------------------------------------------------------+
|Sales      |[alex kim, michael chen, omar hassan]                    |
|HR         |[sara jones, grace lee]                                  |
|Engineering|[john doe, jane smith, priya patel, li wei, daniel brown]|
+-----------+---------------------------------------------------------+



## 21. PIVOT

In [15]:
from pyspark.sql.functions import year

df_year = df.withColumn("join_year", year("join_date"))
df_year.groupBy("dept").pivot("join_year").agg(_sum("salary")).show()

+-----------+--------+-------+-------+-------+--------+-------+
|       dept|    2018|   2019|   2020|   2021|    2022|   2023|
+-----------+--------+-------+-------+-------+--------+-------+
|      Sales|    NULL|   NULL|   NULL|58000.0|124000.0|   NULL|
|         HR|    NULL|   NULL|57000.0|   NULL|    NULL|54000.0|
|Engineering|120000.0|95000.0|88000.0|95000.0|    NULL|79000.0|
+-----------+--------+-------+-------+-------+--------+-------+



## 22. WHEN - OTHERWISE

In [16]:
from pyspark.sql.functions import when

df.withColumn(
    "salary_band",
    when(col("salary") < 60000, "Low")
    .when((col("salary") >= 60000) & (col("salary") < 90000), "Mid")
    .otherwise("High"),
).select("name", "salary", "salary_band").show()

+------------+--------+-----------+
|        name|  salary|salary_band|
+------------+--------+-----------+
|    john doe| 95000.0|       High|
|  jane smith| 88000.0|        Mid|
|    alex kim| 62000.0|        Mid|
| priya patel| 95000.0|       High|
|michael chen| 58000.0|        Low|
|  sara jones| 54000.0|        Low|
|      li wei|120000.0|       High|
| omar hassan| 62000.0|        Mid|
|   grace lee| 57000.0|        Low|
|daniel brown| 79000.0|        Mid|
+------------+--------+-----------+



## 23. JOINS (inner / left / right / full / anti / semi)

In [17]:
emp = spark.read.format("csv").option("header", True) \
    .schema("emp_id INT, name STRING, dept_id INT, salary DOUBLE, join_date DATE") \
    .load("../data/employees_dept_id.csv")

dept = spark.read.format("csv").option("header", True) \
    .schema("dept_id INT, dept STRING, location STRING") \
    .load("../data/departments.csv")

print("INNER -- only matching dept_ids (dept_id=99 and Marketing both drop out):")
emp.join(dept, "dept_id", "inner").select("name", "dept", "location").show()

print("LEFT -- all employees kept, dept_id=99 -> nulls for dept/location:")
emp.join(dept, "dept_id", "left").select("name", "dept", "location").show()

print("RIGHT -- all departments kept, Marketing has no employees -> nulls:")
emp.join(dept, "dept_id", "right").select("name", "dept", "location").show()

print("FULL -- union of both, nulls on whichever side has no match:")
emp.join(dept, "dept_id", "full").select("name", "dept", "location").show()

print("LEFT_ANTI -- employees whose dept_id has NO match in departments:")
emp.join(dept, "dept_id", "left_anti").select("name", "dept_id").show()

print("LEFT_SEMI -- employees WITH a match, only emp's own columns:")
emp.join(dept, "dept_id", "left_semi").select("name", "dept_id").show()

INNER -- only matching dept_ids (dept_id=99 and Marketing both drop out):


+------------+-----------+--------+
|        name|       dept|location|
+------------+-----------+--------+
|    john doe|Engineering| Seattle|
|  jane smith|Engineering| Seattle|
|    alex kim|      Sales|  Austin|
| priya patel|Engineering| Seattle|
|michael chen|      Sales|  Austin|
|  sara jones|         HR| Chicago|
|      li wei|Engineering| Seattle|
| omar hassan|      Sales|  Austin|
|   grace lee|         HR| Chicago|
|daniel brown|Engineering| Seattle|
+------------+-----------+--------+

LEFT -- all employees kept, dept_id=99 -> nulls for dept/location:


+------------+-----------+--------+
|        name|       dept|location|
+------------+-----------+--------+
|    john doe|Engineering| Seattle|
|  jane smith|Engineering| Seattle|
|    alex kim|      Sales|  Austin|
| priya patel|Engineering| Seattle|
|michael chen|      Sales|  Austin|
|  sara jones|         HR| Chicago|
|      li wei|Engineering| Seattle|
| omar hassan|      Sales|  Austin|
|   grace lee|         HR| Chicago|
|daniel brown|Engineering| Seattle|
| fatima noor|       NULL|    NULL|
+------------+-----------+--------+

RIGHT -- all departments kept, Marketing has no employees -> nulls:


+------------+-----------+--------+
|        name|       dept|location|
+------------+-----------+--------+
|daniel brown|Engineering| Seattle|
|      li wei|Engineering| Seattle|
| priya patel|Engineering| Seattle|
|  jane smith|Engineering| Seattle|
|    john doe|Engineering| Seattle|
| omar hassan|      Sales|  Austin|
|michael chen|      Sales|  Austin|
|    alex kim|      Sales|  Austin|
|   grace lee|         HR| Chicago|
|  sara jones|         HR| Chicago|
|        NULL|  Marketing|New York|
+------------+-----------+--------+

FULL -- union of both, nulls on whichever side has no match:


+------------+-----------+--------+
|        name|       dept|location|
+------------+-----------+--------+
|    john doe|Engineering| Seattle|
|  jane smith|Engineering| Seattle|
| priya patel|Engineering| Seattle|
|      li wei|Engineering| Seattle|
|daniel brown|Engineering| Seattle|
|    alex kim|      Sales|  Austin|
|michael chen|      Sales|  Austin|
| omar hassan|      Sales|  Austin|
|  sara jones|         HR| Chicago|
|   grace lee|         HR| Chicago|
|        NULL|  Marketing|New York|
| fatima noor|       NULL|    NULL|
+------------+-----------+--------+

LEFT_ANTI -- employees whose dept_id has NO match in departments:
+-----------+-------+
|       name|dept_id|
+-----------+-------+
|fatima noor|     99|
+-----------+-------+

LEFT_SEMI -- employees WITH a match, only emp's own columns:


+------------+-------+
|        name|dept_id|
+------------+-------+
|    john doe|      1|
|  jane smith|      1|
|    alex kim|      2|
| priya patel|      1|
|michael chen|      2|
|  sara jones|      3|
|      li wei|      1|
| omar hassan|      2|
|   grace lee|      3|
|daniel brown|      1|
+------------+-------+



## 24. WINDOW FUNCTIONS

`row_number()` vs `rank()` vs `dense_rank()`, a running total, and overriding the frame clause.

In [18]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, rank, dense_rank

rank_spec = Window.partitionBy("dept").orderBy(col("salary").desc())

ranked = df.withColumn("row_num", row_number().over(rank_spec)) \
    .withColumn("rank", rank().over(rank_spec)) \
    .withColumn("dense_rank", dense_rank().over(rank_spec))

ranked.select("dept", "name", "salary", "row_num", "rank", "dense_rank") \
    .orderBy("dept", "row_num").show()

print("Second-highest paid employee per dept (rank == 2, so ties are both included):")
ranked.filter(col("rank") == 2).select("dept", "name", "salary").show()

+-----------+------------+--------+-------+----+----------+
|       dept|        name|  salary|row_num|rank|dense_rank|
+-----------+------------+--------+-------+----+----------+
|Engineering|      li wei|120000.0|      1|   1|         1|
|Engineering|    john doe| 95000.0|      2|   2|         2|
|Engineering| priya patel| 95000.0|      3|   2|         2|
|Engineering|  jane smith| 88000.0|      4|   4|         3|
|Engineering|daniel brown| 79000.0|      5|   5|         4|
|         HR|   grace lee| 57000.0|      1|   1|         1|
|         HR|  sara jones| 54000.0|      2|   2|         2|
|      Sales|    alex kim| 62000.0|      1|   1|         1|
|      Sales| omar hassan| 62000.0|      2|   1|         1|
|      Sales|michael chen| 58000.0|      3|   3|         2|
+-----------+------------+--------+-------+----+----------+

Second-highest paid employee per dept (rank == 2, so ties are both included):


+-----------+-----------+-------+
|       dept|       name| salary|
+-----------+-----------+-------+
|Engineering|   john doe|95000.0|
|Engineering|priya patel|95000.0|
|         HR| sara jones|54000.0|
+-----------+-----------+-------+



In [19]:
# Cumulative sum with the DEFAULT frame (unbounded preceding -> current row)
running_spec = Window.partitionBy("dept").orderBy("join_date")
with_running_total = df.withColumn("running_total", _sum("salary").over(running_spec))
with_running_total.select("dept", "name", "join_date", "salary", "running_total") \
    .orderBy("dept", "join_date").show()

# Explicit frame clause -- whole partition, so every row in a dept sees the SAME total
dept_total_spec = (
    Window.partitionBy("dept")
    .orderBy("join_date")
    .rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
)
with_dept_total = df.withColumn("dept_total_salary", _sum("salary").over(dept_total_spec))
with_dept_total.select("dept", "name", "salary", "dept_total_salary") \
    .orderBy("dept").show()

+-----------+------------+----------+--------+-------------+
|       dept|        name| join_date|  salary|running_total|
+-----------+------------+----------+--------+-------------+
|Engineering|      li wei|2018-09-01|120000.0|     120000.0|
|Engineering| priya patel|2019-11-20| 95000.0|     215000.0|
|Engineering|  jane smith|2020-07-01| 88000.0|     303000.0|
|Engineering|    john doe|2021-03-15| 95000.0|     398000.0|
|Engineering|daniel brown|2023-05-23| 79000.0|     477000.0|
|         HR|   grace lee|2020-12-01| 57000.0|      57000.0|
|         HR|  sara jones|2023-02-14| 54000.0|     111000.0|
|      Sales|michael chen|2021-06-05| 58000.0|      58000.0|
|      Sales|    alex kim|2022-01-10| 62000.0|     120000.0|
|      Sales| omar hassan|2022-08-19| 62000.0|     182000.0|
+-----------+------------+----------+--------+-------------+



+-----------+------------+--------+-----------------+
|       dept|        name|  salary|dept_total_salary|
+-----------+------------+--------+-----------------+
|Engineering|      li wei|120000.0|         477000.0|
|Engineering| priya patel| 95000.0|         477000.0|
|Engineering|  jane smith| 88000.0|         477000.0|
|Engineering|    john doe| 95000.0|         477000.0|
|Engineering|daniel brown| 79000.0|         477000.0|
|         HR|   grace lee| 57000.0|         111000.0|
|         HR|  sara jones| 54000.0|         111000.0|
|      Sales|michael chen| 58000.0|         182000.0|
|      Sales|    alex kim| 62000.0|         182000.0|
|      Sales| omar hassan| 62000.0|         182000.0|
+-----------+------------+--------+-----------------+



## 25. User Defined Functions (UDF vs pandas_udf)

In [20]:
from pyspark.sql.functions import udf, pandas_udf
from pyspark.sql.types import StringType
import pandas as pd

@udf(returnType=StringType())
def categorize_salary(salary):
    if salary is None:
        return "Unknown"
    return "High" if salary > 80000 else "Low"

df.withColumn("salary_category", categorize_salary(col("salary"))) \
    .select("name", "salary", "salary_category").show()

@pandas_udf(StringType())
def categorize_salary_pandas(salary: pd.Series) -> pd.Series:
    return salary.apply(lambda s: "Unknown" if pd.isna(s) else ("High" if s > 80000 else "Low"))

df.withColumn("salary_category", categorize_salary_pandas(col("salary"))) \
    .select("name", "salary", "salary_category").show()

/home/claude/pyspark-interview-prep/venv/lib/python3.11/site-packages/pyspark/sql/udf.py:116: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


+------------+--------+---------------+
|        name|  salary|salary_category|
+------------+--------+---------------+
|    john doe| 95000.0|           High|
|  jane smith| 88000.0|           High|
|    alex kim| 62000.0|            Low|
| priya patel| 95000.0|           High|
|michael chen| 58000.0|            Low|
|  sara jones| 54000.0|            Low|
|      li wei|120000.0|           High|
| omar hassan| 62000.0|            Low|
|   grace lee| 57000.0|            Low|
|daniel brown| 79000.0|            Low|
+------------+--------+---------------+

+------------+--------+---------------+
|        name|  salary|salary_category|
+------------+--------+---------------+
|    john doe| 95000.0|           High|
|  jane smith| 88000.0|           High|
|    alex kim| 62000.0|            Low|
| priya patel| 95000.0|           High|
|michael chen| 58000.0|            Low|
|  sara jones| 54000.0|            Low|
|      li wei|120000.0|           High|
| omar hassan| 62000.0|            Low|

/home/claude/pyspark-interview-prep/venv/lib/python3.11/site-packages/pyspark/sql/pandas/functions.py:777: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/home/claude/pyspark-interview-prep/venv/lib/python3.11/site-packages/pyspark/sql/pandas/typehints.py:60: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


## 26-27. Writing data, write modes, and Parquet

In [21]:
import shutil, os

out_path = "../data/output/employees_parquet"
shutil.rmtree(out_path, ignore_errors=True)

df.write.format("parquet").mode("overwrite").partitionBy("dept").save(out_path)

reloaded = spark.read.format("parquet").load(out_path)  # no schema/inferSchema needed -- Parquet embeds it
reloaded.printSchema()
reloaded.show(3)

print("Files on disk (partitioned by dept):")
for root, dirs, files in os.walk(out_path):
    for f in files:
        if f.endswith(".parquet"):
            print(os.path.join(root, f))

root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- salary: double (nullable = true)
 |-- join_date: date (nullable = true)
 |-- skills: string (nullable = true)
 |-- dept: string (nullable = true)

+------+-----------+-------+----------+------------------+-----------+
|emp_id|       name| salary| join_date|            skills|       dept|
+------+-----------+-------+----------+------------------+-----------+
|   101|   john doe|95000.0|2021-03-15|  Python,Spark,SQL|Engineering|
|   102| jane smith|88000.0|2020-07-01|        Java,Spark|Engineering|
|   104|priya patel|95000.0|2019-11-20|Python,SQL,Airflow|Engineering|
+------+-----------+-------+----------+------------------+-----------+
only showing top 3 rows
Files on disk (partitioned by dept):
../data/output/employees_parquet/dept=Engineering/part-00000-16abc0aa-f3d8-45a4-8922-c8eb3504d410.c000.snappy.parquet
../data/output/employees_parquet/dept=HR/part-00000-16abc0aa-f3d8-45a4-8922-c8eb3504d410.c00

In [22]:
# append vs overwrite vs error vs ignore
append_path = "../data/output/append_demo"
shutil.rmtree(append_path, ignore_errors=True)

df.limit(2).write.format("parquet").mode("overwrite").save(append_path)
print("after first write:", spark.read.parquet(append_path).count(), "rows")

df.limit(2).write.format("parquet").mode("append").save(append_path)
print("after append:", spark.read.parquet(append_path).count(), "rows  (doubled)")

df.limit(2).write.format("parquet").mode("ignore").save(append_path)
print("after ignore (no-op, data already exists):", spark.read.parquet(append_path).count(), "rows  (unchanged)")

try:
    df.limit(2).write.format("parquet").mode("error").save(append_path)
except Exception as e:
    print("mode='error' raised as expected:", type(e).__name__)

after first write: 2 rows


after append: 4 rows  (doubled)


after ignore (no-op, data already exists): 4 rows  (unchanged)
mode='error' raised as expected: AnalysisException


## 28. Managed vs External Tables

Requires a Hive-compatible metastore to run `CREATE TABLE` for real. Shown here as reference SQL (see README §28 for the full explanation) --
not executed against this local session, which doesn't have a metastore configured.

In [23]:
managed_table_sql = '''
CREATE TABLE managed_employees (emp_id INT, name STRING, salary DOUBLE)
'''  # no LOCATION -> Spark/the metastore owns the data; DROP TABLE deletes data + metadata

external_table_sql = '''
CREATE TABLE external_employees (emp_id INT, name STRING, salary DOUBLE)
USING PARQUET
LOCATION '/mnt/data/employees/'
'''  # LOCATION given -> you own the data; DROP TABLE deletes metadata ONLY

print(managed_table_sql)
print(external_table_sql)


CREATE TABLE managed_employees (emp_id INT, name STRING, salary DOUBLE)


CREATE TABLE external_employees (emp_id INT, name STRING, salary DOUBLE)
USING PARQUET
LOCATION '/mnt/data/employees/'



## 29. Spark SQL (createOrReplaceTempView + spark.sql)

In [24]:
df.createOrReplaceTempView("employees")

result = spark.sql('''
    SELECT dept, ROUND(AVG(salary), 2) AS avg_salary, COUNT(*) AS headcount
    FROM employees
    GROUP BY dept
    ORDER BY avg_salary DESC
''')
result.show()

+-----------+----------+---------+
|       dept|avg_salary|headcount|
+-----------+----------+---------+
|Engineering|   95400.0|        5|
|      Sales|  60666.67|        3|
|         HR|   55500.0|        2|
+-----------+----------+---------+



## 32. Lazy evaluation, transformations vs actions -- proof by log inspection

In [25]:
# Building a chain of transformations does NOT execute anything yet
step1 = df.filter(col("salary") > 50000)
step2 = step1.select("name", "dept")
step3 = step2.withColumn("dept_upper", upper("dept"))

print("Plan built. Nothing has run. Job count so far:",
      spark.sparkContext.statusTracker().getJobIdsForGroup())

step3.show(3)  # <- THIS action triggers execution of the whole chain at once

print("Job count after the action:",
      len(spark.sparkContext.statusTracker().getJobIdsForGroup() or []))

Plan built. Nothing has run. Job count so far: [85, 84, 83, 82, 81, 80, 79, 78, 77, 76, 75, 74, 73, 72, 71, 70, 69, 68, 67, 66, 65, 64, 63, 62, 61, 60, 59, 58, 57, 56, 55, 54, 53, 52, 51, 50, 49, 48, 47, 46, 45, 44, 43, 42, 41, 40, 39, 38, 37, 36, 35, 34, 33, 32, 31, 30, 29, 28, 27, 26, 25, 24, 23, 22, 21, 20, 19, 18, 17, 16, 15, 14, 13, 12, 11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1, 0]
+----------+-----------+-----------+
|      name|       dept| dept_upper|
+----------+-----------+-----------+
|  john doe|Engineering|ENGINEERING|
|jane smith|Engineering|ENGINEERING|
|  alex kim|      Sales|      SALES|
+----------+-----------+-----------+
only showing top 3 rows
Job count after the action: 87


## 33. Partitioning: repartition vs coalesce

In [26]:
print("original partitions:", df.rdd.getNumPartitions())

df_more = df.repartition(6)
print("after repartition(6):", df_more.rdd.getNumPartitions())

df_fewer = df_more.coalesce(2)
print("after coalesce(2):", df_fewer.rdd.getNumPartitions())

df_by_key = df.repartition(3, "dept")
print("after repartition(3, 'dept'):", df_by_key.rdd.getNumPartitions())

original partitions: 1


after repartition(6): 6
after coalesce(2): 2
after repartition(3, 'dept'): 3


## 34. Cache vs persist

In [27]:
from pyspark import StorageLevel

cached_df = df.filter(col("salary") > 0).cache()
cached_df.count()  # action that materializes the cache
print("is cached:", cached_df.is_cached)

cached_df.unpersist()
print("is cached after unpersist:", cached_df.is_cached)

is cached: True
is cached after unpersist: False


## 35. Broadcast join

In [28]:
from pyspark.sql.functions import broadcast

broadcast_joined = emp.join(broadcast(dept), "dept_id", "inner")
broadcast_joined.select("name", "dept", "location").show()

# Inspect the physical plan -- look for "BroadcastHashJoin" in the output
broadcast_joined.explain()

+------------+-----------+--------+
|        name|       dept|location|
+------------+-----------+--------+
|    john doe|Engineering| Seattle|
|  jane smith|Engineering| Seattle|
|    alex kim|      Sales|  Austin|
| priya patel|Engineering| Seattle|
|michael chen|      Sales|  Austin|
|  sara jones|         HR| Chicago|
|      li wei|Engineering| Seattle|
| omar hassan|      Sales|  Austin|
|   grace lee|         HR| Chicago|
|daniel brown|Engineering| Seattle|
+------------+-----------+--------+

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [dept_id#799, emp_id#797, name#798, salary#800, join_date#801, dept#803, location#804]
   +- BroadcastHashJoin [dept_id#799], [dept_id#802], Inner, BuildRight, false, false
      :- Filter isnotnull(dept_id#799)
      :  +- FileScan csv [emp_id#797,name#798,dept_id#799,salary#800,join_date#801] Batched: false, DataFilters: [isnotnull(dept_id#799)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/claude/pyspark-i

## 36. Catalyst / physical plan inspection

In [29]:
df.filter(col("salary") > 80000).select("name", "dept").explain(True)

== Parsed Logical Plan ==
'Project ['name, 'dept]
+- Filter (salary#3 > cast(80000 as double))
   +- Relation [emp_id#0,name#1,dept#2,salary#3,join_date#4,skills#5] csv

== Analyzed Logical Plan ==
name: string, dept: string
Project [name#1, dept#2]
+- Filter (salary#3 > cast(80000 as double))
   +- Relation [emp_id#0,name#1,dept#2,salary#3,join_date#4,skills#5] csv

== Optimized Logical Plan ==
Project [name#1, dept#2]
+- Filter (isnotnull(salary#3) AND (salary#3 > 80000.0))
   +- Relation [emp_id#0,name#1,dept#2,salary#3,join_date#4,skills#5] csv

== Physical Plan ==
*(1) Project [name#1, dept#2]
+- *(1) Filter (isnotnull(salary#3) AND (salary#3 > 80000.0))
   +- FileScan csv [name#1,dept#2,salary#3] Batched: false, DataFilters: [isnotnull(salary#3), (salary#3 > 80000.0)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/claude/pyspark-interview-prep/data/employees.csv], PartitionFilters: [], PushedFilters: [IsNotNull(salary), GreaterThan(salary,80000.0)], ReadSchema: str

## 37-38. Delta Lake and Structured Streaming (reference only)

These need the `delta-spark` package and (for streaming) a running source -- not executed in this
lightweight demo notebook. See README.md sections 37 and 38 for the full, correct code:
- `DeltaTable.merge()` for upserts / SCD Type 2
- `spark.read.format("delta").option("versionAsOf", n)` for time travel
- `spark.readStream` / `.writeStream` with a checkpoint location for streaming aggregations

## Cleanup

In [30]:
spark.stop()